# Enzyme Function Prediction using Neural Networks (EC Level 3)

In this homework, we will design several neural networks with different architectures for **Enzyme Function Prediction** from protein sequences.

**Enzyme Commission (EC) numbers** constitute a numerical classification scheme for enzymes, based on the chemical reactions they catalyze. Unlike protein families (Pfam) which are often defined by sequence similarity, EC numbers describe the **functional role** of a protein (e.g., *what* reaction it performs). This makes the task challenging, as the model must learn complex structure-function relationships where different sequences might catalyze similar reactions (convergent evolution).

In this problem, each protein in the training or test set is labeled with exactly one **Level-3 EC number** derived from the **UniProtKB/Swiss-Prot** database. The EC number is a hierarchical string (e.g., `2.7.1.-`), where:

* **Level 1**: Main Class (e.g., `2` = Transferases)
* **Level 2**: Subclass (e.g., `2.7` = Transferring phosphorus-containing groups)
* **Level 3**: Sub-subclass (Target of this task) (e.g., `2.7.1` = Phosphotransferases with an alcohol group as acceptor)



There are a total of 183 distinct functional categories (Level-3) appearing in the dataset. Although many enzymes are "promiscuous" (catalyzing multiple different reactions), we simplify the problem by filtering for single-function enzymes and treating it as a single-label multi-class classification task.

To begin with, run the following cell to download the training and test data.

> ### Resource Warning: OOM & Slow Training
>
> You may encounter **`CUDA out of memory`** errors or find the training process slow due to the size of the models.
>
> **For Students:** Google Colab offers **free compute units** for users with student email addresses. This provides access to high-RAM GPUs such as A100, R100.
> **[Click here to apply for Student Compute Units](https://colab.research.google.com/signup?utm_source=resource_tab&utm_medium=link&utm_campaign=want_more_resources)**

Import the necessary packages:

In [169]:
# export
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, os, time
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import math
##### DO NOT MODIFY ANYTHING IN THIS CELL #####

Load the training data and take a look at the sequences and EC number labels

In [170]:
with open('labels.json') as f:
    label_list = json.load(f)

# Create mapping from Label string to Integer index
label2idx = {label: idx for idx, label in enumerate(label_list)}
print(f'Number of Subcellular Locations: {len(label_list)}')

Number of Subcellular Locations: 183


In [171]:
df_train = pd.read_csv('train.csv')
sequences = df_train['Sequence'].tolist()
labels = df_train['Label'].tolist()
df_train.head()

,Entry,Sequence,EC number,Label
0,Q61176,MSSKPKSLEIIGAPFSKGQPRGGVEKGPAALRKAGLLEKLKETEYD...,3.5.3.1,3.5.3
1,Q21MC3,MTDSKSNSNSPSLSYKDAGVDIDAGNALVERIKSVAKRTRRPEVMA...,6.3.3.1,6.3.3
2,Q85WU9,MAHSVKIYDTCIGCTQCVRACPTDVLEMIPWEGCKAKQIASAPRTE...,1.97.1.12,1.97.1
3,B7GGC6,MFMSQFHATTIFAIRHQGKGAMAGDGQVTFGNAVVMKHTARKIRKL...,3.4.25.2,3.4.25
4,Q32067,KRSLRDLGIPVNQIIPEGGSLKYLKDLPRAWFNIVPYREVGLMTAI...,1.3.7.7,1.3.7


In [172]:
df_val = pd.read_csv('val.csv')
seq_val = df_val['Sequence'].tolist()
label_val = df_val['Label'].tolist()
df_val.head()

,Entry,Sequence,EC number,Label
0,Q16595,MWTLGRRAVAGLLASPSPAQAQTLTRVPRPAELAPLCGRRGLRTDI...,1.16.3.1,1.16.3
1,Q1CHE5,MQLNIPTWLTLFRVVLIPFFVLAFYLPFVWAPMVCAIIFVFAAATD...,2.7.8.5,2.7.8
2,P60586,MDFNLNDEQELFVAGIRELMASENWEAYFAECDRDSVYPERFVKAL...,1.3.8.13,1.3.8
3,Q99TL8,MNGVYHIMNNEYPYSADEVLHKAKSYLSADEYEYVLKSYHIAYEAH...,2.7.6.5,2.7.6
4,Q9X794,MVAPLAEVDPDIAELLGKELGRQRDTLEMIASENFVPRSVLQAQGS...,2.1.2.1,2.1.2


In [173]:
print(f'Training samples: {len(sequences)}')
print(f'Validation samples: {len(seq_val)}')

Training samples: 18094
Validation samples: 3877


In [174]:
print("num_classes from json:", len(label_list))
print("train label examples:", df_train["Label"].head(10).tolist())

num_classes from json: 183
train label examples: ['3.5.3', '6.3.3', '1.97.1', '3.4.25', '1.3.7', '1.14.19', '2.7.2', '4.99.1', '3.1.13', '1.20.4']


## Task 1: One-hot tokenizer

Protein sequences consist of a list of amino acids. There are 20 types of standard amino acids. We need to transform (tokenize) protein sequences into tensors so that neural networks can take them as inputs. A straightforward way to tokenize protein sequences is to use one-hot encoding ([wiki link](https://en.wikipedia.org/wiki/One-hot)). In this task you need to complete the function `one_hot_encode` which takes a protein sequence (a string of amino acids) of length $L$ and output an one-hot-encoded tensor of shape $L\times 20$.

In [175]:
# export
##### DO NOT MODIFY ANYTHING ABOVE THIS LINE #####

amino_acids = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

# One-hot encoding function
def one_hot_encode(sequence, max_len=1000):
    sequence = sequence[:max_len]
    encoded = torch.zeros((len(sequence), 20))
    ######################################################################
    # TODO: Implement one-hot encoding loop
    # for i in range(len(sequence)):
    #     string_pos = amino_acids.find(sequence[i])
    #     encoded[i, string_pos] = 1
    idx = torch.tensor([aa_to_idx[aa] for aa in sequence if aa in aa_to_idx])
    encoded = F.one_hot(idx, num_classes=20).float()
    ######################################################################

    return encoded

In [176]:
test_seq = 'MWTLGRRAVAGLLASPSPAQAQTLTRVPRPAELAPLCGRRGLRTDI'
print(one_hot_encode(test_seq))

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
         0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.],
        [0

With the one-hot tokenizer, we can design the dataset class. As you can see in the following cell, each data point returned by the dataset class contains three items: the one-hot encoded tensor, the length of the sequence, and the label. Since the length of the sequences in the dataset can vary, we provide the collate function that pads the sequences with zeros to the maximum length in the batch. You can pass this `collate_fn` to the `collate_fn` parameter of pytorch's DataLoader class to ensure the correct behaviour of batching.

In [177]:
# export
##### DO NOT MODIFY ANYTHING ABOVE THIS LINE #####

class ProteinDataset(Dataset):
    def __init__(self, sequences, labels, ec2idx):
        self.sequences = sequences
        self.labels = [ec2idx.get(ec, -1) for ec in labels]
        self.seq_tensors = [one_hot_encode(seq) for seq in tqdm(sequences, desc='Encoding sequences')]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq_tensor = self.seq_tensors[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return seq_tensor, len(seq_tensor), label

# Collate function with padding
def collate_fn(batch):
    sequences, seq_lens, labels = zip(*batch)
    max_len = max(seq_lens)

    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)

    return padded_sequences, torch.tensor(seq_lens, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

## Task 2: Transformer built from multi-head attention

In this task, you are required to implement a vanilla transformer encoder model for the EC number prediction task. You should construct the transformer model using blocks like `torch.nn.MultiheadAttention`, `torch.nn.Linear`, `torch.nn.LayerNorm`. Your transformer model should have the same architecture as the encoder module described in the paper [Attention is all you need](https://arxiv.org/abs/1706.03762). We recommend you to check PyTorch's documentation for the modules mentioned before.

## Task 2.1: Positional Encoding

Protein sequences are naturally ordered, and the relative position of amino acids often dictates the functional domain of an enzyme. However, the Multi-head Attention mechanism used in Transformers is "permutation-invariant," meaning it does not inherently understand the order of the input. To address this, we must inject **Positional Encoding (PE)** into the input embeddings.

Following the architecture in *Attention Is All You Need*, we use sine and cosine functions of different frequencies to generate unique position vectors:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

Where:
* **$pos$**: The position of the amino acid in the sequence.
* **$i$**: The dimension index.
* **$d_{model}$**: The dimensionality of the hidden state (embedding dimension).



In this task, you need to implement the `PositionalEncoding` module.

In [178]:
# export
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        """
        Args:
            d_model: The dimension of the embeddings.
            max_len: The maximum length of the sequences.
        """
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        ######################################################################
        # TODO: Implement the frequency term 'div_term'.
        # Hint: Use the formula exp(arange(0, d_model, 2) * -(log(10000.0) / d_model))
        # TODO: Populate the 'pe' matrix.
        # Apply sine to even indices (0::2) and cosine to odd indices (1::2).
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0)) / d_model)
        pe[:, 0::2] = torch.sin(position*div_term)
        pe[:, 1::2] = torch.cos(position*div_term)
        ######################################################################

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        """
        Args:
            x: Input tensor of shape [batch_size, seq_len, d_model]
        """
        x = x + self.pe[:, :x.size(1), :]
        return x

## Task 2.2 Attention Classifier

In [179]:
class AttentionClassifier(nn.Module):
    def __init__(self, num_classes, embed_dim=64, num_heads=1, num_layers=1, ff_dim=128, dropout_p=0.1):
        super(AttentionClassifier, self).__init__()
        self.embedding = nn.Linear(20, embed_dim)
        self.attention_layers = None
        ######################################################################
        # TODO: implement the transformer block using multiheadattention, linear, and layernorm.
        self.pos_encoder = PositionalEncoding(embed_dim)
        self.dropout = nn.Dropout(p=dropout_p)

        self.attention_layers = nn.ModuleList()
        for _ in range(num_layers):
            attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
            ln1 = nn.LayerNorm(embed_dim)
            linear1 = nn.Linear(embed_dim, ff_dim)
            relu = nn.ReLU()
            linear2 = nn.Linear(ff_dim, embed_dim)
            ln2 = nn.LayerNorm(embed_dim)

            self.attention_layers.append(nn.ModuleList([attn, ln1, linear1, relu, linear2, ln2]))
        ######################################################################
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x, seq_lens):
        x = self.embedding(x)
        x = self.pos_encoder(x) # You can call the positional encoding function here
        x = self.dropout(x)
        max_len = x.shape[1]
        mask = torch.arange(max_len, device=x.device).expand(len(seq_lens), max_len) >= seq_lens.unsqueeze(1)
        mask = mask.to(x.device)

        for attn, ln1, linear1, relu, linear2, ln2 in self.attention_layers:
            attn_out, _ = attn(x, x, x, key_padding_mask=mask)
            attn_out = self.dropout(attn_out)
            x = ln1(x + attn_out)
            ff_out = linear2(relu(linear1(x)))
            ff_out = self.dropout(ff_out)
            x = ln2(x + ff_out)

        x = x.permute(0, 2, 1)
        x = self.pooling(x).squeeze(-1)

        return self.fc(x)

## Task 3: Transformer built from TransformerEncoder class

In this task, you will also implement a vanilla transformer model. Instead of constructing the model from small blocks like MultiheadAttention, you should use the wrapped module `torch.nn.TransformerEncoderLayer` and `torch.nn.TransformerEncoder` to directly build the model. We recommend you to check the documentation of these two modules to learn their usage.

In [180]:
class TransformerClassifier(nn.Module):
    def __init__(self, num_classes, embed_dim=128, num_heads=2, num_layers=2, ff_dim=256, dropout_p=0.1):
        super(TransformerClassifier, self).__init__()
        self.embedding = nn.Linear(20, embed_dim)
        self.encoder_layers = None
        ######################################################################
        # TODO: transformer block using TransformerEncoderLayer
        self.pos_encoder = PositionalEncoding(embed_dim)

        self.encoder_layers = nn.ModuleList()
        for _ in range(num_layers):
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=embed_dim, 
                nhead=num_heads, 
                dim_feedforward=ff_dim,
                batch_first=True,
                dropout=dropout_p)
            self.encoder_layers.append(encoder_layer)
        # transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        ######################################################################
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x, seq_lens):
        x = self.embedding(x)
        x = self.pos_encoder(x) # You can call the positional encoding function here
        max_len = x.shape[1]
        mask = torch.arange(max_len, device=x.device).expand(len(seq_lens), max_len) >= seq_lens.unsqueeze(1)
        # print(mask.shape)
        mask = mask.to(x.device)
        ######################################################################
        # TODO: forward part of the transformer block
        for layer in self.encoder_layers:
            x = layer.forward(x, src_key_padding_mask=mask)
        ######################################################################
        x = x.permute(0, 2, 1)
        x = self.pooling(x).squeeze(-1)

        return self.fc(x)

## Task 4: 1D-CNN model

In this task, you are going to implement a model using 1D CNN layers. You can use PyTorch's `torch.nn.Conv1d` to construct the model. Note that for simplicity, you do not have to consider the padded part of the input tensor. Refer to PyTorch's documentation for the usage of `torch.nn.Conv1d`.

In [203]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes, embed_dim=64, num_filters=128, kernel_size=3, num_layers=3):
        super(CNNClassifier, self).__init__()
        self.embedding = nn.Linear(20, embed_dim)
        self.conv_layers = None
        ######################################################################
        # TODO: 1D convolutional layers
        self.conv_layers = nn.ModuleList()
        conv_layer_1 = nn.Conv1d(embed_dim, num_filters, kernel_size)

        self.conv_layers.append(conv_layer_1)
        self.conv_layers.append(nn.ReLU())

        for _ in range(num_layers-1):
            self.conv_layers.append(nn.Conv1d(num_filters, num_filters, kernel_size))
            self.conv_layers.append(nn.ReLU())
        ######################################################################
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(num_filters, num_classes)

    def forward(self, x, seq_lens):
        x = self.embedding(x).permute(0, 2, 1)
        ######################################################################
        # TODO: forward part of the CNN block
        for layer in self.conv_layers:
            x = layer.forward(x)
        ######################################################################
        x = self.pooling(x).squeeze(-1)
        return self.fc(x)

## Training the Neural Networks

Blocks marked with `# export` are required by the autograder and should not be modified.  
However, you are strongly encouraged to experiment with **all other code blocks**.

Examples of blocks you can safely adjust:
- Model initialization (e.g., hidden size, number of layers)
- The training loop (optimizer, learning rate, scheduler, gradient handling)
- The `train_model(...)` call and its arguments
- Data-related setup (batch size)

---

### Hyperparameter Tuning

You are encouraged to tune hyperparameters to improve validation performance.  
Common categories include:

**Optimization**
- Optimizer type  
- Learning rate and scheduler  
- Weight decay  

**Training setup**
- Batch size  
- Number of epochs  
- Early stopping  

**Model capacity**
- Hidden dimensions  
- Number of layers  
- Attention heads (Transformer)  
- CNN channels / kernel sizes  

**Regularization & stability**
- Dropout rates  
- Label smoothing  
- Gradient clipping (e.g., `clip_grad_norm_`)

If training becomes unstable (e.g., exploding gradients), gradient clipping may help.  

---

### Model Design Hints

- **Positional Encoding**: Helps the attention mechanism understand amino acid order, which is important for sequential structure modeling.

- **Dropout**: Applied after embedding and inside feed-forward blocks. Tuning dropout helps balance underfitting and overfitting.

- **Model Size**: Increasing embedding dimension or feed-forward size increases capacity but also memory usage.

- **Classification Head**: A multi-layer head (`Linear → ReLU → Dropout → Linear`) can be more expressive than a single linear layer.

- **Key Padding Mask**: Prevents attention from focusing on padded tokens, which is important for variable-length protein sequences.


**Be mindful of GPU memory — if you encounter OOM errors, reduce `batch_size` first before shrinking the model.**

Complete the function `train_model`.

In [182]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim

In [198]:
def train_model(model, train_dataset, val_dataset, num_classes, epochs=100, batch_size=256, lr=1e-3, patience=10, device='cuda:0', wd=0.01):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_fn)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

    best_acc = 0
    patience_counter = 0
    best_ckpt = None

    for epoch in range(epochs):
        start_epoch = time.time()
        model.train()
        total_loss, correct, total = 0, 0, 0

        for sequences, seq_lens, labels in train_loader:
            sequences, seq_lens, labels = sequences.to(device), seq_lens.to(device), labels.to(device)
            outputs = None
            loss = None
            ######################################################################
            # TODO: Implement the training step
            optimizer.zero_grad()
            outputs = model.forward(sequences, seq_lens)
            loss = criterion(outputs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, norm_type=2.0)
            optimizer.step()
            ######################################################################

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for sequences, seq_lens, labels in val_loader:
                ######################################################################
                # TODO: model inference
                sequences, seq_lens, labels = sequences.to(device), seq_lens.to(device), labels.to(device)
                outputs = model(sequences, seq_lens)
                preds = torch.argmax(outputs, dim=1)
                ######################################################################
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = correct / total
        scheduler.step(val_acc)

        end_epoch = time.time()
        print(f'Epoch [{epoch+1} / {epochs}]: Train Loss={total_loss:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, LR={optimizer.param_groups[0]["lr"]:.4f}, Time={end_epoch - start_epoch:.4f} sec')

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            best_ckpt = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    return model, best_ckpt

### Train AttentionClassifier

In [199]:
train_dataset = ProteinDataset(sequences, labels, label2idx) # Use the lists loaded from train.csv
val_dataset = ProteinDataset(seq_val, label_val, label2idx)  # Use the lists loaded from val.csv
num_classes = len(label_list)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = AttentionClassifier(
    num_classes,
    embed_dim=64,
    num_heads=2,
    num_layers=1,
    ff_dim=128,
    dropout_p=0.1
).to(device)

model, best_ckpt = train_model(model, train_dataset, val_dataset, num_classes=num_classes, epochs=100, batch_size=16, lr=5e-3, patience=15, device=device, wd=1e-4)
model.load_state_dict(best_ckpt)

df_test = pd.read_csv('test.csv')
test_sequences = df_test['Sequence'].tolist()
test_seq2name = {seq: f'test_seq_{i}' for i, seq in enumerate(test_sequences)}
test_dataset = ProteinDataset(test_sequences, [label_list[0]]*len(test_sequences), label2idx)
test_loader = DataLoader(test_dataset, batch_size=64, collate_fn=collate_fn)

model.eval()
preds = []
with torch.no_grad():
    for batch_seqs, batch_lens, _ in test_loader:
        ######################################################################
        # TODO: inference on the test set
        batch_seqs, batch_lens = batch_seqs.to(device), batch_lens.to(device)
        outputs = model(batch_seqs, batch_lens)
        preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())
        ######################################################################
# save the predictions to a individual CSV file
preds = [label_list[pred] for pred in preds]
df_preds = pd.DataFrame(preds)
df_preds.to_csv('test_preds_attention.csv', index=False, header=False)

Encoding sequences: 100%|██████████| 3877/3877 [00:00<00:00, 8680.06it/s]


Epoch [1 / 100]: Train Loss=5300.7364, Train Acc=0.0652, Val Acc=0.1034, LR=0.0050, Time=11.3576 sec
Epoch [2 / 100]: Train Loss=4697.7063, Train Acc=0.1615, Val Acc=0.1888, LR=0.0050, Time=11.1555 sec
Epoch [3 / 100]: Train Loss=4320.9661, Train Acc=0.2481, Val Acc=0.2688, LR=0.0050, Time=11.2945 sec
Epoch [4 / 100]: Train Loss=4085.4930, Train Acc=0.2999, Val Acc=0.3283, LR=0.0050, Time=11.2071 sec
Epoch [5 / 100]: Train Loss=3911.1972, Train Acc=0.3351, Val Acc=0.3490, LR=0.0050, Time=11.1071 sec
Epoch [6 / 100]: Train Loss=3762.0151, Train Acc=0.3668, Val Acc=0.3799, LR=0.0050, Time=11.4221 sec
Epoch [7 / 100]: Train Loss=3643.3813, Train Acc=0.3915, Val Acc=0.3949, LR=0.0050, Time=11.3778 sec
Epoch [8 / 100]: Train Loss=3522.4348, Train Acc=0.4190, Val Acc=0.4238, LR=0.0050, Time=11.5112 sec
Epoch [9 / 100]: Train Loss=3420.7105, Train Acc=0.4400, Val Acc=0.4297, LR=0.0050, Time=11.4845 sec
Epoch [10 / 100]: Train Loss=3321.6127, Train Acc=0.4655, Val Acc=0.4573, LR=0.0050, Time=1

Encoding sequences: 100%|██████████| 3878/3878 [00:00<00:00, 5843.58it/s]


In [200]:
'''
Best trials:  
Trial 21 finished with value: 0.46762961052360075 and parameters: {'learning_rate': 0.004550336419726336, 'num_heads': 1, 'num_layers': 2, 'dropout_p': 0.10328973400525898, 'wd': 0.027927639305451207}
Trial 26 finished with value: 0.47175651276760383 and parameters: {'learning_rate': 0.005088370821326657, 'num_heads': 1, 'num_layers': 2, 'dropout_p': 0.10106683010377945, 'wd': 0.013552521785811001}
'''
# print(study.best_trial.params)

"\nBest trials:  \nTrial 21 finished with value: 0.46762961052360075 and parameters: {'learning_rate': 0.004550336419726336, 'num_heads': 1, 'num_layers': 2, 'dropout_p': 0.10328973400525898, 'wd': 0.027927639305451207}\nTrial 26 finished with value: 0.47175651276760383 and parameters: {'learning_rate': 0.005088370821326657, 'num_heads': 1, 'num_layers': 2, 'dropout_p': 0.10106683010377945, 'wd': 0.013552521785811001}\n"

### Train TransformerClassifier

In [201]:
model = TransformerClassifier(num_classes,
                embed_dim=64,
                num_heads=2,
                num_layers=1,
                ff_dim=128,
                dropout_p=0.1).to(device)
model, best_ckpt = train_model(model, train_dataset, val_dataset, num_classes=num_classes, epochs=120, batch_size=30, lr=5e-3, patience=15, device=device, wd=1e-4)
model.load_state_dict(best_ckpt)

model.eval()
preds = []
with torch.no_grad():
    for batch_seqs, batch_lens, _ in test_loader:
        ######################################################################
        # TODO: inference on the test set
        batch_seqs, batch_lens = batch_seqs.to(device), batch_lens.to(device)
        outputs = model(batch_seqs, batch_lens)
        preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())
        ######################################################################

preds = [label_list[pred] for pred in preds]
df_preds = pd.DataFrame(preds)
df_preds.to_csv('test_preds_transformer.csv', index=False, header=False)

Epoch [1 / 120]: Train Loss=2829.9706, Train Acc=0.0666, Val Acc=0.1060, LR=0.0050, Time=8.9753 sec
Epoch [2 / 120]: Train Loss=2547.3009, Train Acc=0.1591, Val Acc=0.2017, LR=0.0050, Time=9.0760 sec
Epoch [3 / 120]: Train Loss=2311.5083, Train Acc=0.2502, Val Acc=0.2824, LR=0.0050, Time=9.0919 sec
Epoch [4 / 120]: Train Loss=2126.0736, Train Acc=0.3245, Val Acc=0.3474, LR=0.0050, Time=9.1242 sec
Epoch [5 / 120]: Train Loss=1989.8681, Train Acc=0.3806, Val Acc=0.3959, LR=0.0050, Time=9.0610 sec
Epoch [6 / 120]: Train Loss=1877.8649, Train Acc=0.4348, Val Acc=0.4256, LR=0.0050, Time=9.1381 sec
Epoch [7 / 120]: Train Loss=1785.5960, Train Acc=0.4713, Val Acc=0.4403, LR=0.0050, Time=9.1405 sec
Epoch [8 / 120]: Train Loss=1718.0780, Train Acc=0.5020, Val Acc=0.4712, LR=0.0050, Time=9.1364 sec
Epoch [9 / 120]: Train Loss=1655.0864, Train Acc=0.5287, Val Acc=0.5022, LR=0.0050, Time=9.2119 sec
Epoch [10 / 120]: Train Loss=1605.9725, Train Acc=0.5486, Val Acc=0.5267, LR=0.0050, Time=9.2514 sec

In [133]:
# def objective_transformer(trial):
#     torch.cuda.empty_cache() # Keep mem usage low

#     batch_size = 30
#     learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
#     # embed_dim = trial.suggest_int("embed_dim", 64, 128, step=16)
#     num_heads = trial.suggest_int("num_heads", 1, 2, step=1)
#     num_layers = trial.suggest_int("num_layers", 1, 2, step=1)
#     # ff_dim = trial.suggest_int("ff_dim", 128, 256, step=32)
#     dropout_p = trial.suggest_float("dropout_p", 0.1, 0.5)
#     wd = trial.suggest_float("wd", 0.01, 0.1) # Weight decay for AdamW
    
#     model = TransformerClassifier(num_classes,
#                 embed_dim=64,
#                 num_heads=num_heads,
#                 num_layers=num_layers,
#                 ff_dim=128).to(device),
#                 dropout_p=dropout_p
#     model, best_ckpt = train_model(model, train_dataset, val_dataset, num_classes=num_classes, epochs=100, batch_size=batch_size, lr=learning_rate, patience=7, device=device)
#     model.load_state_dict(best_ckpt)
    
#     # Evaluate on validation set to get the metric
#     model.load_state_dict(best_ckpt)
#     model.eval()
#     correct, total = 0, 0
#     val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate_fn)
#     with torch.no_grad():
#         for seqs, lens, labels in val_loader:
#             seqs, lens, labels = seqs.to(device), lens.to(device), labels.to(device)
#             outputs = model(seqs, lens)
#             correct += (outputs.argmax(1) == labels).sum().item()
#             total += labels.size(0)
#     val_acc = correct / total
#     return val_acc  # maximize this

# study = optuna.create_study(
#     direction="maximize",
#     pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2))
# study.optimize(objective_attn, n_trials=10)



# model = TransformerClassifier(num_classes,
#                 embed_dim=64,
#                 num_heads=1,
#                 num_layers=1,
#                 ff_dim=128).to(device)
# model, best_ckpt = train_model(model, train_dataset, val_dataset, num_classes=num_classes, epochs=100, batch_size=64, lr=1e-3, patience=7, device=device)
# model.load_state_dict(best_ckpt)

# model.eval()
# preds = []
# with torch.no_grad():
#     for batch_seqs, batch_lens, _ in test_loader:
#         ######################################################################
#         # TODO: inference on the test set
#         batch_seqs, batch_lens = batch_seqs.to(device), batch_lens.to(device)
#         outputs = model(batch_seqs, batch_lens)
#         preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())
#         ######################################################################

# preds = [label_list[pred] for pred in preds]
# df_preds = pd.DataFrame(preds)
# df_preds.to_csv('test_preds_transformer.csv', index=False, header=False)

### Train CNNClassifier

In [ ]:
model = CNNClassifier(num_classes,
            embed_dim=64,
            num_filters=128,
            kernel_size=3,
            num_layers=3).to(device)
model, best_ckpt = train_model(model, train_dataset, val_dataset, num_classes=num_classes, epochs=100, batch_size=64, lr=1e-3, patience=7, device=device)
model.load_state_dict(best_ckpt)

model.eval()
preds = []
with torch.no_grad():
    for batch_seqs, batch_lens, _ in test_loader:
        ######################################################################
        # TODO: inference on the test set
        batch_seqs, batch_lens = batch_seqs.to(device), batch_lens.to(device)
        outputs = model(batch_seqs, batch_lens)
        preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())
        ######################################################################
# save the predictions to a individual CSV file
preds = [label_list[pred] for pred in preds]
df_preds = pd.DataFrame(preds)
df_preds.to_csv('test_preds_cnn.csv', index=False, header=False)

Epoch [1 / 100]: Train Loss=1457.4141, Train Acc=0.0077, Val Acc=0.0142, Time=3.5675 sec
Epoch [2 / 100]: Train Loss=1406.8795, Train Acc=0.0146, Val Acc=0.0155, Time=3.3706 sec
Epoch [3 / 100]: Train Loss=1362.4697, Train Acc=0.0304, Val Acc=0.0426, Time=3.3705 sec
Epoch [4 / 100]: Train Loss=1320.0871, Train Acc=0.0437, Val Acc=0.0536, Time=3.3750 sec
Epoch [5 / 100]: Train Loss=1305.6066, Train Acc=0.0475, Val Acc=0.0500, Time=3.4049 sec
Epoch [6 / 100]: Train Loss=1296.4492, Train Acc=0.0558, Val Acc=0.0562, Time=3.3668 sec
Epoch [7 / 100]: Train Loss=1277.2232, Train Acc=0.0694, Val Acc=0.0702, Time=3.4056 sec
Epoch [8 / 100]: Train Loss=1264.9611, Train Acc=0.0766, Val Acc=0.0838, Time=3.4001 sec
Epoch [9 / 100]: Train Loss=1256.9035, Train Acc=0.0802, Val Acc=0.0800, Time=3.4389 sec
Epoch [10 / 100]: Train Loss=1250.5169, Train Acc=0.0850, Val Acc=0.0776, Time=3.3559 sec
Epoch [11 / 100]: Train Loss=1242.5940, Train Acc=0.0905, Val Acc=0.0898, Time=3.4526 sec
Epoch [12 / 100]: T

## Task 5: Using pretrained protein language model embeddings

In the previous tasks we are using one-hot encoded sequences as the model inputs. With the advancement of language models, many pretrained protein language models (pLM) have been widely used in protein-related problems. Below we are going to explore the usage of pLM embeddings for EC number prediction. We will use ESM-2 (https://github.com/facebookresearch/esm) to extract protein sequence embeddings. First you need to to check ESM-2's documentation to learn how to generate the embeddings using the fasta files we have provided. You should use the model `esm2_t33_650M_UR50D`, retrieve the last-layer sequence-level embedding (no need for residue-level embedding). You should generate one `.pt` file for each sequence embedding and save it in the directory `esm_embeddings`. Since the embedding generation can be time-consuming, we will use a subsampled training set (50% of the original training set) for this task. If you have adquate computational resources, you can also use the complete training set.

In [ ]:
!pip install fair-esm
!pip install biopython


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import esm
from Bio import SeqIO
from tqdm.auto import tqdm
import os
import numpy as np
import torch

def gen_emb(fasta_file, out_dir='esm_embeddings', device='cuda:0'):
    os.makedirs(out_dir, exist_ok=True)
    records = list(SeqIO.parse(fasta_file, 'fasta'))
    names = [rec.id for rec in records]
    sequences = [str(rec.seq) for rec in records]
    print(f'Number of sequences: {len(sequences)}')

    data = [(name, seq[:1000]) for name, seq in zip(names, sequences)] # Truncate to 1000 residues to shorten runtime

    ######################################################################
    # TODO: Load ESM-2 model (esm2_t33_650M_UR50D) and batch converter
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    ######################################################################

    model.to(device)
    model.eval()
    model.half() # Half-prec. to shorten runtime

    batch_size = 4 # Reduce if you are running out of cuda memory
    num_batches = int(np.ceil(len(data) / batch_size))

    for i in tqdm(range(num_batches)):
        batch = data[i * batch_size:(i + 1) * batch_size]
        names_batch, seqs_batch = zip(*batch)

        # Skip batch if all embeddings already exist
        # if all(os.path.exists(os.path.join(out_dir, f'{n}.pt')) for n in names_batch):
        #     continue

        batch_labels, batch_strs, batch_tokens = batch_converter(batch)
        batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
        batch_tokens = batch_tokens.to(device)
        
        # Extract per-residue representations (on CPU); use torch.autocast to handle 16-bit prec.
        with torch.no_grad(), torch.autocast(device_type="cuda"):
            results = model(batch_tokens, repr_layers=[33])
        token_representations = results["representations"][33].detach().cpu()
        
        # Generate per-sequence representations via averaging
        for k, tokens_len in enumerate(batch_lens):
            seq_name = names_batch[k]
            out_path = os.path.join(out_dir, f'{seq_name}.pt')

            # Skip batch operations if sequences have already been handled
            # if os.path.exists(out_path):
            #     continue

            seq_tokens = token_representations[k, :tokens_len]
            seq_mean = seq_tokens.mean(0)
            torch.save(seq_mean, out_path)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5070 Laptop GPU


In [ ]:
gen_emb('train_subsample.fasta', out_dir='esm_embeddings_train')

Number of sequences: 3619


  0%|          | 0/905 [00:00<?, ?it/s]

In [ ]:
gen_emb('test_seqs.fasta', out_dir='esm_embeddings_test')

Number of sequences: 3878


  0%|          | 0/970 [00:00<?, ?it/s]

In [ ]:
df_sub = pd.read_csv('train_subsample.csv')

from Bio import SeqIO
train_sub_records = list(SeqIO.parse('train_subsample.fasta', 'fasta'))
train_sub_seq2name = {str(rec.seq): rec.id for rec in train_sub_records}

sequences_sub = df_sub['Sequence'].tolist()
labels_sub = df_sub['Label'].tolist()

seq_train_sub, seq_val_sub, label_train_sub, label_val_sub = train_test_split(
    sequences_sub, labels_sub, test_size=0.2, random_state=42
)

Construct a new dataset class to use ESM-2 embeddings.

In [ ]:
class ProteinESMDataset(Dataset):
    def __init__(self, sequences, seq2name, emb_dir, labels, ec2idx):
        super().__init__()
        self.labels = [ec2idx.get(ec, -1) for ec in labels]
        self.embeddings = []
        for seq in tqdm(sequences, desc='Loading esm embeddings'):
            name = seq2name[seq]
            emb_file = os.path.join(emb_dir, f'{name}.pt')
            emb = torch.load(emb_file)
            self.embeddings.append(emb)

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, index):
        emb = self.embeddings[index]
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return emb, label

Implement a simple MLP model to use ESM-2 embeddings as inputs for EC number prediction.

In [147]:
class MLPClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=1280, hidden_dim=640, dropout_p=0.1):
        super(MLPClassifier, self).__init__()
        ######################################################################
        # TODO: linear layers
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, num_classes)
        )
        ######################################################################

    def forward(self, x):
        ######################################################################
        # TODO: forward function
        ######################################################################
        return self.layers(x)

Complete the function `train_model_esm`, which trains the model taking ESM-2 embeddings as inputs.

In [148]:
def train_model_esm(model, train_dataset, val_dataset, num_classes, epochs=100, batch_size=256, lr=1e-3, patience=10, device='cuda:0'):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_acc = 0
    patience_counter = 0
    best_ckpt = None

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = None
            loss = None
            ######################################################################
            # TODO: backpropogation
            optimizer.zero_grad()
            outputs = model.forward(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            ######################################################################

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

        train_acc = correct / total

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for sequences, labels in val_loader:
                ######################################################################
                # TODO: inference
                sequences, labels = sequences.to(device), labels.to(device)
                outputs = model(sequences)
                preds = torch.argmax(outputs, dim=1)
                ######################################################################
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = correct / total
        print(f'Epoch {epoch+1}: Train Loss={total_loss:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}')

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            best_ckpt = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    return model, best_ckpt

Train the MLP model and generate predictions for the test set.

In [150]:
emb_dir = 'esm_embeddings'

test_sub_records = list(SeqIO.parse('test_seqs.fasta', 'fasta'))
test_sub_seq2name = {str(rec.seq): rec.id for rec in test_sub_records}

train_dataset_esm = ProteinESMDataset(seq_train_sub, train_sub_seq2name, 'esm_embeddings_train', label_train_sub, label2idx)
val_dataset_esm = ProteinESMDataset(seq_val_sub, train_sub_seq2name, 'esm_embeddings_train', label_val_sub, label2idx)

model_esm = MLPClassifier(num_classes, input_dim=1280, hidden_dim=640,dropout_p=0.3).to(device)
model_esm, best_ckpt_esm = train_model_esm(
    model_esm, train_dataset_esm, val_dataset_esm,
    num_classes=num_classes, epochs=500, batch_size=256, lr=1e-4, patience=50, device=device
)
model_esm.load_state_dict(best_ckpt_esm)

df_test = pd.read_csv('test.csv')
test_sequences = df_test['Sequence'].tolist()

test_dataset_esm = ProteinESMDataset(test_sequences, test_sub_seq2name, 'esm_embeddings_test', [label_list[0]]*len(test_sequences), label2idx)
test_loader_esm = DataLoader(test_dataset_esm, batch_size=256)

model_esm.eval()
preds = []
with torch.no_grad():
    for sequences, _ in test_loader_esm:
        ######################################################################
        # TODO: inference on the test set
        sequences = sequences.to(device)
        outputs = model_esm(sequences)
        preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())
        ######################################################################


preds = [label_list[pred] for pred in preds]
df_preds = pd.DataFrame(preds)
df_preds.to_csv('test_preds_esm.csv', index=False, header=False)

Loading esm embeddings: 100%|██████████| 724/724 [00:00<00:00, 1484.74it/s]


Epoch 1: Train Loss=62.0692, Train Acc=0.0214, Val Acc=0.0124
Epoch 2: Train Loss=57.5481, Train Acc=0.1071, Val Acc=0.0442
Epoch 3: Train Loss=53.9190, Train Acc=0.2370, Val Acc=0.1519
Epoch 4: Train Loss=50.6255, Train Acc=0.3244, Val Acc=0.3246
Epoch 5: Train Loss=47.5264, Train Acc=0.4152, Val Acc=0.4337
Epoch 6: Train Loss=44.8990, Train Acc=0.4750, Val Acc=0.4779
Epoch 7: Train Loss=42.3273, Train Acc=0.5254, Val Acc=0.5138
Epoch 8: Train Loss=39.9393, Train Acc=0.5658, Val Acc=0.5345
Epoch 9: Train Loss=37.8289, Train Acc=0.5941, Val Acc=0.5428
Epoch 10: Train Loss=35.5429, Train Acc=0.6269, Val Acc=0.5594
Epoch 11: Train Loss=33.5719, Train Acc=0.6663, Val Acc=0.5787
Epoch 12: Train Loss=31.6192, Train Acc=0.6846, Val Acc=0.5912
Epoch 13: Train Loss=29.9637, Train Acc=0.7150, Val Acc=0.6257
Epoch 14: Train Loss=28.2419, Train Acc=0.7292, Val Acc=0.6298
Epoch 15: Train Loss=26.7810, Train Acc=0.7541, Val Acc=0.6340
Epoch 16: Train Loss=25.4890, Train Acc=0.7748, Val Acc=0.6519
E

Loading esm embeddings: 100%|██████████| 3878/3878 [00:00<00:00, 5013.46it/s]


## Grading

### Task 1 (4 points)
Task 1 will be graded by the correctness of the function `one_hot_encode`.

### Task 2 (9 points)
Tasks 2-5 will be graded by the accuracy of the predictions made by each model on the test set.

- Accuracy >= 0.7, 9 points
- Accuracy >= 0.65, 8.5 points
- Accuracy >= 0.6, 7 points
- Accuracy >= 0.55, 6 points
- Accuracy < 0.55, 0 points

### Task 3 (9 points)

- Accuracy >= 0.7, 9 points
- Accuracy >= 0.65, 8.5 points
- Accuracy >= 0.6, 7 points
- Accuracy >= 0.55, 6 points
- Accuracy < 0.55, 0 points

### Task 4 (9 points)

- Accuracy >= 0.55, 9 points
- Accuracy >= 0.50, 7 points
- Accuracy >= 0.45, 5 points
- Accuracy < 0.45, 0 points

### Task 5 (9 points)

- Accuracy >= 0.85, 9 points
- Accuracy >= 0.8, 7 points
- Accuracy >= 0.75, 5 points
- Accuracy < 0.75, 0 points

## Submission

After completing all the tasks, you need to submit five files to Gradescope:
- `hw2_nn.ipynb`, the notebook file with all tasks completed.
- `test_preds_attention.csv`
- `test_preds_transformer.csv`
- `test_preds_cnn.csv`
- `test_preds_esm.csv`

Note that the four `.csv` files will be automatically generated when running this notebook, do not change the codes regarding the save of the prediction results. **DO NOT** submit the files in a zip file, please submit them individually to Gradescope.